## Intro

## Initialization

Start notebook server with uv run --with jupyter jupyter lab


## Byte Pair Encoding (BPE)

Byte Pair Encoding (BPE) is a subword tokenization algorithm commonly used in natural language processing. It iteratively replaces the most frequent pair of bytes or characters in a text with a single, unused symbol. This process continues until a predefined vocabulary size is reached.

**Key Steps in BPE:**
1. Start with a base vocabulary of all unique characters in the dataset.
2. Count the frequency of all adjacent character pairs.
3. Merge the most frequent pair into a new symbol.
4. Repeat steps 2–3 until the desired vocabulary size is achieved.

**Advantages:**
- Handles out-of-vocabulary words by breaking them into known subword units.
- Reduces vocabulary size while maintaining the ability to represent rare and compound words.

**Applications:**
- Widely used in tokenization for neural machine translation and large language models (e.g., GPT, BERT).

In [1]:
import nltk

text = "That U.S.A. poster-print costs $12.40..."

pattern = r"""(?x)
(                               # Outer capturing group for the full match
    (?:[A-Z]\.)+                # Abbreviations like U.S.A.
    | \w+(?:-\w+)* # Words with optional hyphens
    | \$?\d+(?:\.\d+)?%?        # Currency and percentages, e.g. $12.40, 82%
    | \.\.\.                    # Ellipsis
    | [\]\[.,;"'?():_\-`]       # Punctuation (escaped hyphen, backtick included)
)
"""

result = nltk.regexp_tokenize(text, pattern)
print(result)


['That', 'U.S.A.', 'poster-print', 'costs', '$12.40', '...']


# BPE Utility functions

In [2]:
import re  # added for digit mapping
import string
from collections import defaultdict

from tqdm import tqdm


def split_text(text):
    text_len = len(text)
    part_size = text_len // 100
    train_size = (text_len - part_size) // 2
    train_set_1 = text[:train_size]
    test_set = text[train_size : train_size + part_size]
    train_set_2 = text[train_size + part_size :]
    train_set = train_set_1 + train_set_2
    return train_set, test_set


def normalize_text(text, remove_punctuation=True):
    text = text.lower()  # Convert to lowercase
    text = " ".join(text.split())  # Remove extra spaces
    text = text.replace("\n", " ")  # Replace newlines with spaces
    if remove_punctuation:
        text = text.translate(
            str.maketrans("", "", string.punctuation)
        )  # Remove punctuation
    return text


def advanced_normalize(
    text: str,
    keep_sentence_final: bool = False,
    keep_apostrophes: bool = False,
    map_digits: bool = True,
) -> str:
    """Advanced normalization applying three optional strategies:
    1. Selective punctuation retention (keep .?! and/or apostrophes) while removing others.
    2. Quote / ellipsis normalization: fancy quotes -> ' or ", ellipsis … -> three dots.
    3. Digit mapping: map all digits 0-9 to 0 to reduce sparsity.

    Args:
        text: Raw input string.
        keep_sentence_final: Keep . ? ! if True.
        keep_apostrophes: Keep apostrophes (') if True.
        map_digits: Replace every digit with 0 if True.
    Returns:
        Normalized string.
    """
    # Normalize whitespace early (preserve baseline behavior order)
    text = text.replace("\n", " ")
    text = " ".join(text.split())
    # Lowercase (reuse baseline assumption)
    text = text.lower()

    # 2. Quote / ellipsis normalization before punctuation stripping
    replacements = {
        "“": '"',
        "”": '"',
        "„": '"',
        "′": "'",
        "’": "'",
        "‘": "'",
        "‛": "'",
        "…": "...",
    }
    text = "".join(replacements.get(ch, ch) for ch in text)

    # 1. Selective punctuation removal
    # Build punctuation set to remove
    allowed = set()
    if keep_sentence_final:
        allowed.update([".", "!", "?"])
    if keep_apostrophes:
        allowed.add("'")
    # Remove all other ASCII punctuation
    remove_chars = "".join(ch for ch in string.punctuation if ch not in allowed)
    if remove_chars:
        text = text.translate(str.maketrans("", "", remove_chars))

    # 3. Digit mapping
    if map_digits:
        text = re.sub(r"\d", "0", text)

    # Collapse whitespace again in case removals created doubles
    text = " ".join(text.split())
    return text


def update_vocab(vocab, addition):
    vocab.append(addition)
    return vocab


def prep_text(text):
    words = text.split()
    new_text = []
    for word in words:
        word = word + "_"
        new_text.append(word)

    new_text = " ".join(new_text)
    new_text = list(new_text)
    return new_text


def get_max_pair(text, track_progress=True):
    pairs = defaultdict(int)
    if track_progress:
        pbar = tqdm(total=len(text) - 1, desc="Getting max pairs")
    for i in range(len(text) - 1):
        pair = text[i : i + 2]
        pair = "".join(pair)
        if pair in pairs:
            pairs[pair] += 1
        else:
            pairs[pair] = 1
        if track_progress:
            pbar.update(1)

    # Find the most frequent pair
    most_frequent_pair = max(pairs, key=pairs.get)
    most_frequent_count = pairs[most_frequent_pair]
    return most_frequent_pair, most_frequent_count


def update_text(text, substitute_pair, track_progress=False):
    i = 0
    new_text = []
    if track_progress:
        pbar = tqdm(total=len(text) - 1, desc="Updating text")
    while i < len(text):
        if i == len(text) - 1:
            new_text.append(text[i])
            break
        pair = text[i] + text[i + 1]
        if pair == substitute_pair:
            new_text.append(substitute_pair)
            i += 2  # Skip the next token as it was merged
            # Do not increment i, check for overlapping pairs
        else:
            new_text.append(text[i])
            i += 1
        if track_progress:
            pbar.update(1)
    # if new text is empty return the original text
    if new_text == []:
        return text
    elif new_text == text:
        return text
    else:
        return new_text


def test_vocab(text, vocab=None, k=2000):
    """Test the vocabulary against a new text."""
    # load vocab
    if vocab is None:
        vocab = open(f"data/bpe_outputs/vocab_with_k{k}.txt", "r").read()
    print(f"Testing with vocabulary of size {len(vocab.splitlines())}")
    # prepare text
    text = normalize_text(text)
    words = text.split()
    tokens = 0
    unknown_tokens = []

    for word in words:
        word = word + "_"
        i = 0
        while i < len(word):
            found_token = False
            # Try to find longest matching token
            for j in range(len(word), i, -1):
                if word[i:j] in vocab:
                    tokens += 1
                    i = j
                    found_token = True
                    break
                else:
                    tokens += 1
                    unknown_tokens.append(word[i:j])
                    i = j
            if not found_token:
                unknown_tokens.append(word[i])
                i += 1
                tokens += 1

    coverage = (tokens - len(unknown_tokens)) / tokens * 100

    return coverage, unknown_tokens


def get_token_counts(vocab, corpus):
    counted_vocab = defaultdict(int)
    for token in vocab.splitlines():
        counted_vocab[token] = corpus.count(token)
    return counted_vocab


def reprocess_corpus(corpus, vocab):
    """
    Reprocess the corpus to create a vocabulary of size k.
    Args:
        corpus (str): The input text.
        k (int): The desired vocabulary size.
    Returns:
        str: The processed text with the vocabulary of size k.
    """
    corpus = normalize_text(corpus)
    corpus = list(corpus.replace(" ", "_"))
    for token in vocab.splitlines():
        if len(token) < 2:
            continue
        else:
            update_text(corpus, token, track_progress=False)
    pass


# BPE 

In [ ]:
import string

from tqdm import tqdm


def perform_bpe(
    text="data/corpora/shakespeare.txt",
    k=2000,
    normalization=None,
    track_progress=False,
    save_to=None,
):
    k_start = k
    if save_to == None:
        save_to = f"data/bpe_outputs/vocab_with_k{k_start}.txt"
    # Load and normalize the text
    text = open(text, "r").read()
    if normalization == "advanced":
        text = advanced_normalize(text)
    else:
        text = normalize_text(text)

    # Split the text into training and test sets
    text = list(text.replace(" ", "_"))

    # Create the initial vocabulary
    vocab = list(string.ascii_lowercase) + ["_"]

    pbar = tqdm(total=k, desc="Merging pairs")

    while k > 0:
        # Get the most frequent pair in the text
        if track_progress:
            print("Getting most frequent pair...")
        most_frequent_pair, count = get_max_pair(text, track_progress)

        # If no pairs found, break
        if count < 2:
            break

        # Replace the most frequent pair in the text
        if track_progress:
            print("Updating text with most frequent pair...")
        text = update_text(text, most_frequent_pair, track_progress)

        # Add the new token to the vocabulary
        vocab.append(most_frequent_pair)

        k -= 1
        pbar.update(1)

    with open(save_to, "w", encoding="utf-8") as f:
        for token in vocab:
            f.write(token + "\n")
    # Derive merges (Option B: tokens length > 1)
    merges = [t for t in vocab if len(t) > 1]
    merges_path = save_to.replace("vocab_with_k", "merges_k")
    with open(merges_path, "w", encoding="utf-8") as mf:
        for m in merges:
            mf.write(m + "\n")
    print(f"Vocabulary saved to {save_to} (merges derived -> {merges_path})")
    return save_to


if __name__ == "__main__":
    track_progress = False
    text = open("data/corpora/shakespeare.txt", "r").read()
    text = normalize_text(text)

    train, test = split_text(text)
    # train = train[:100000]
    text = list(train.replace(" ", "_"))
    vocab = list(string.ascii_lowercase) + ["_"]
    k = 1000
    k_start = k

    pbar = tqdm(total=k, desc="Merging pairs")

    while k > 0:
        # Get the most frequent pair in the text
        if track_progress:
            print("Getting most frequent pair...")
        most_frequent_pair, count = get_max_pair(text, track_progress)
        # If no pairs found, break
        if count < 2:
            break

        # Replace the most frequent pair in the text
        if track_progress:
            print("Updating text with most frequent pair...")
        text = update_text(text, most_frequent_pair, track_progress)
        # Add the new token to the vocabulary
        vocab.append(most_frequent_pair)

        k -= 1
        pbar.update(1)

    print("Vocabulary done, saving...")
    # Save the vocabulary to a file
    with open(f"data/bpe_outputs/vocab_with_k{k_start}.txt", "w") as f:
        for token in vocab:
            f.write(token + "\n")
    print("Vocabulary saved.")


# BPE Test

In [ ]:
import os

from src.tokenizer.bpe import perform_bpe
from src.tokenizer.bpe_utils import split_text, test_vocab

try:
    import matplotlib.pyplot as plt

    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False
    print("Matplotlib not available. Visualization will be skipped.")

# --- Added flag for advanced normalization run naming ---
ADVANCED = False  # Set True when using advanced normalization pipeline
ADV_SUFFIX = "_adv" if ADVANCED else ""


def visualize_coverage_results(results):
    """Create visualizations for BPE vocabulary coverage results."""
    if not HAS_MATPLOTLIB:
        print("Matplotlib not available. Skipping visualization.")
        return

    k_values = list(results.keys())
    text_names = list(results[k_values[0]].keys())

    # Create subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

    # Plot 1: Coverage vs Vocabulary Size
    for text_name in text_names:
        coverages = [results[k][text_name]["coverage"] for k in k_values]
        ax1.plot(k_values, coverages, marker="o", linewidth=2, label=text_name)

    ax1.set_xlabel("Vocabulary Size (k)")
    ax1.set_ylabel("Coverage (%)")
    ax1.set_title("BPE Vocabulary Coverage vs Size")
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(60, 100)

    # Plot 2: Unknown Tokens vs Vocabulary Size (log scale)
    for text_name in text_names:
        unknown_counts = [results[k][text_name]["unknown_count"] for k in k_values]
        ax2.plot(k_values, unknown_counts, marker="s", linewidth=2, label=text_name)

    ax2.set_xlabel("Vocabulary Size (k)")
    ax2.set_ylabel("Unknown Tokens (log scale)")
    ax2.set_title("Unknown Tokens vs Vocabulary Size")
    ax2.set_yscale("log")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(
        "data/bpe_outputs/bpe_coverage_analysis.png", dpi=300, bbox_inches="tight"
    )
    plt.show()


def print_coverage_summary(results):
    """Print a formatted summary table of coverage results."""
    k_values = list(results.keys())
    text_names = list(results[k_values[0]].keys())

    print("\n" + "=" * 80)
    print("COVERAGE SUMMARY TABLE")
    print("=" * 80)
    print(
        f"{'Vocab Size':<12} {'Shakespeare':<15} {'Knox Country':<15} {'Room w/ View':<15}"
    )
    print("-" * 80)

    for k in k_values:
        row = f"{k:<12}"
        for text_name in text_names:
            coverage = results[k][text_name]["coverage"]
            row += f"{coverage:>10.1f}%    "
        print(row)


if __name__ == "__main__":
    # Test different vocabulary sizes
    k_values = [50, 500, 1000, 1250, 1500, 2000]
    version = "clean"  # Choose between "clean" or "dirty" or empty string "" for original text

    # Set up file paths based on version
    if version == "clean":
        train_text = "data/corpora/Shakespeare_clean_train.txt"
        val_text = "data/corpora/Shakespeare_clean_valid.txt"
    elif version == "dirty":
        train_text = "data/corpora/shakespeare_dirty_train.txt"
        val_text = "data/corpora/shakespeare_dirty_valid.txt"
    else:
        # Load and normalize the text
        text = open("data/corpora/shakespeare.txt", "r").read()
        train_text, val_text = split_text(text)

    # Print which version  and files are being used
    print(f"Using '{version}' version of Shakespeare text.")
    print(f"Training text file: {train_text}")
    print(f"Validation text file: {val_text}")
    print(f"Advanced flag: {ADVANCED} (vocab files will have suffix '{ADV_SUFFIX}')")

    # Test texts to evaluate vocabulary coverage
    test_texts = [
        ("Shakespeare Validation", val_text),
        ("In Mr. Knox's Country", "data/corpora/In Mr. Knox's Country.txt"),
        ("A Room with a View", "data/corpora/A Room with a View.txt"),
    ]

    print("=" * 60)
    print("BPE Vocabulary Coverage Test")
    print("=" * 60)

    # Store results for visualization
    coverage_results = {k: {} for k in k_values}

    for k in k_values:
        print(f"\n--- Testing with k={k} ---")

        vocab_filename = f"data/bpe_outputs/vocab_with_k{k}{ADV_SUFFIX}.txt"
        # Check if vocabulary exists, create if not
        if os.path.exists(vocab_filename):
            print(f"Using existing vocabulary file: {vocab_filename}")
            vocab_location = vocab_filename
        else:
            print(f"Running BPE tokenizer for k={k} (creating {vocab_filename})...")
            vocab_location = perform_bpe(
                text=train_text,
                k=k,
                track_progress=False,
                save_to=vocab_filename,
            )

        # Test vocabulary on different texts
        for text_name, text_path in test_texts:
            print(f"\nTesting {text_name}:")

            if text_name == "Shakespeare Validation":
                text_content = open(text_path, "r", encoding="utf-8").read()
            else:
                text_content = open(text_path, "r", encoding="utf-8").read()

            vocab_content = open(vocab_location, "r", encoding="utf-8").read()

            coverage, unknown_tokens = test_vocab(
                text_content, vocab=vocab_content, k=k
            )
            print(f"  Coverage: {coverage:.2f}%")
            if len(unknown_tokens) > 0:
                print(f"  Unknown tokens (first 10): {list(unknown_tokens)[:10]}")
                print(f"  Total unknown tokens: {len(unknown_tokens)}")

            coverage_results[k][text_name] = {
                "coverage": coverage,
                "unknown_count": len(unknown_tokens),
            }

    print_coverage_summary(coverage_results)
    visualize_coverage_results(coverage_results)

    print("\n" + "=" * 60)
    print("Test complete!")
